# Vimeo-90K chunked training on Colab

Automates: download one ~6-10GB Vimeo-90K chunk from Kaggle (`wangsally/vimeo-90k-1` .. `-10`) -> train `BaselineAutoencoder` on it -> checkpoint to Google Drive -> delete the chunk -> move to the next chunk. Repeats for every chunk listed in `CHUNK_NUMBERS`.

**Before running:**
- A Kaggle account + API token (kaggle.com/settings -> API -> Create New Token). You upload it once; it is then cached on your Drive so future reconnects do not ask again.
- A Google account for Drive (checkpoints and progress live there so a disconnect does not lose anything).

**If the session disconnects:** reopen this notebook and Runtime -> Run all again. Already-downloaded/trained chunks are recorded in `progress.json` on Drive and are skipped automatically; training resumes from the last saved checkpoint.

**At the end:** the last cell downloads `best.pt` straight to your machine.

**Design notes (full reasoning in the chat this notebook came from):**
- Frames are never copied off the Kaggle chunk into a second location - each chunk's folders are symlinked directly into `data/external/vimeo_septuplet/sequences/`, so local disk use stays close to one chunk's size (~6-10GB) at a time, never the full ~89GB.
- The train/test split written for each chunk (`sep_trainlist.txt` / `sep_testlist.txt`) is a fresh, reproducible split *of that chunk only*. It exists purely to give training a validation signal per chunk, and is discarded when the chunk is deleted. It is **not** the official Vimeo-90K split, and it is **not** meant to be a final benchmark.
- To compare this model against one trained on the full dataset elsewhere (e.g. a friend's machine), evaluate both on a fixed, independent set neither model trained on - DAVIS's existing test split (already in this repo, untouched by anything here) is a good candidate for that.

In [ ]:
from pathlib import Path

# --- Which Kaggle chunks to cycle through (wangsally/vimeo-90k-1 .. vimeo-90k-10) ---
CHUNK_NUMBERS = list(range(1, 11))  # e.g. [1, 2, 3] to do fewer if you are short on time/disk

# --- Training ---
EPOCHS_PER_CHUNK = 2     # epochs run on each chunk before moving to the next
BATCH_SIZE = 32
LATENT_CHANNELS = None   # None -> use configs/default.json's value. Keep this THE SAME across every
                          # run you resume from - changing it will break loading an existing checkpoint.
LEARNING_RATE = None     # None -> use configs/default.json's value
CROP_SIZE = 256          # Vimeo frames are 448x256 natively; crop size must be divisible by 16
SEED = 42
NUM_WORKERS = 2          # Colab is Linux (fork-based) - safe to use >0, unlike the Windows default of 0

# --- Where things live ---
REPO_URL = 'https://github.com/yuvidewan/neural_streaming.git'
REPO_DIR = Path('/content/neural_streaming')
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/neural_streaming_colab')
LOCAL_SCRATCH_DIR = Path('/content/vimeo_scratch')  # wiped and recreated per chunk, never accumulates

DELETE_CHUNK_AFTER_TRAINING = True
KAGGLE_DATASET_OWNER = 'wangsally'
KAGGLE_DATASET_PREFIX = 'vimeo-90k'  # -> wangsally/vimeo-90k-1, vimeo-90k-2, ...

## 1. Mount Google Drive

Checkpoints and progress state are written here, not to Colab's local disk, so nothing is lost on disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = DRIVE_PROJECT_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
PROGRESS_PATH = DRIVE_PROJECT_DIR / 'progress.json'
print(f'Drive project dir: {DRIVE_PROJECT_DIR}')
print(f'Checkpoints will be saved to: {CHECKPOINT_DIR}')

## 2. Kaggle API credentials

Uploaded once via the file picker below, then cached on Drive - future reconnects reuse the cached copy automatically and will not prompt again.

In [ ]:
import os
import shutil
from pathlib import Path

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_json_target = kaggle_dir / 'kaggle.json'
drive_kaggle_json = DRIVE_PROJECT_DIR / 'kaggle.json'

if drive_kaggle_json.is_file():
    shutil.copy(drive_kaggle_json, kaggle_json_target)
    print(f'Reused Kaggle API token from {drive_kaggle_json}')
else:
    print('No saved Kaggle token found on Drive - upload your kaggle.json now.')
    print('(Get it from https://www.kaggle.com/settings -> API -> Create New Token)')
    from google.colab import files
    uploaded = files.upload()
    uploaded_name = next(iter(uploaded))
    shutil.move(uploaded_name, kaggle_json_target)
    shutil.copy(kaggle_json_target, drive_kaggle_json)
    print(f'Saved a copy to {drive_kaggle_json} - future reconnects will reuse it automatically.')

os.chmod(kaggle_json_target, 0o600)

## 3. Get the project code

Clones `neural_streaming` from GitHub and installs it editable, so `import nvc` uses the exact same code as your (and your collaborator's) local checkouts.

In [ ]:
import os
import subprocess

if REPO_DIR.is_dir():
    print(f'{REPO_DIR} already exists - pulling latest')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print(f'cwd: {os.getcwd()}')

In [ ]:
!pip install -q -r requirements.txt
!pip install -e .
!pip install -q kaggle

In [ ]:
# Fails loudly right here if the editable install above didn't actually take -
# better than a confusing ModuleNotFoundError several cells later.
import nvc
print(f'nvc package loaded OK from: {nvc.__file__}')

## 4. Imports and setup

In [ ]:
import json
import random
import shutil
import subprocess
import zipfile
from pathlib import Path

import torch

from nvc.data.loaders import create_sequence_test_loader, create_sequence_train_loader
from nvc.data.vimeo import build_sequence_manifest
from nvc.models import BaselineAutoencoder
from nvc.training import resume_training_state, save_checkpoint, train_one_epoch, validate_one_epoch
from nvc.utils.config import load_default_config
from nvc.utils.device import get_device
from nvc.utils.seed import seed_everything

defaults = load_default_config()
device = get_device()
seed_everything(SEED)

VIMEO_ROOT = defaults.vimeo_root
VIMEO_ROOT.mkdir(parents=True, exist_ok=True)

latent_channels = LATENT_CHANNELS or defaults.latent_channels
learning_rate = LEARNING_RATE or defaults.learning_rate
print(f'VIMEO_ROOT: {VIMEO_ROOT}')
print(f'latent_channels={latent_channels}  learning_rate={learning_rate}')

## 5. Helper functions

Chunk download/extraction, zero-copy symlink merge into `VIMEO_ROOT/sequences`, per-chunk split-list generation, and manifest building via the existing `nvc.data.vimeo` pipeline.

In [ ]:
def _find_sequences_source_root(chunk_dir: Path) -> Path:
    '''Locate the folder whose children are Vimeo "<group>" directories, by
    finding the first im1.png anywhere under chunk_dir and walking up two
    levels (im1.png -> <clip>/ -> <group>/ -> the folder holding all groups).
    '''
    for im1 in chunk_dir.rglob('im1.png'):
        clip_dir = im1.parent
        group_dir = clip_dir.parent
        return group_dir.parent
    raise RuntimeError(f'No im1.png found anywhere under {chunk_dir} - unexpected chunk layout')


def download_and_extract_chunk(chunk_number: int, scratch_dir: Path) -> Path:
    '''Download one wangsally/vimeo-90k-N Kaggle dataset and extract it.
    Returns the folder whose direct children are Vimeo "<group>" folders.
    '''
    if scratch_dir.exists():
        shutil.rmtree(scratch_dir)
    scratch_dir.mkdir(parents=True)

    slug = f'{KAGGLE_DATASET_OWNER}/{KAGGLE_DATASET_PREFIX}-{chunk_number}'
    print(f'[chunk {chunk_number}] downloading {slug} ...')
    subprocess.run(['kaggle', 'datasets', 'download', '-d', slug, '-p', str(scratch_dir)], check=True)

    zips = list(scratch_dir.glob('*.zip'))
    if not zips:
        raise RuntimeError(f'[chunk {chunk_number}] no .zip downloaded into {scratch_dir}')
    print(f'[chunk {chunk_number}] extracting {zips[0].name} ...')
    with zipfile.ZipFile(zips[0]) as zf:
        zf.extractall(scratch_dir)
    zips[0].unlink()

    return _find_sequences_source_root(scratch_dir)


def relink_sequences_to_chunk(chunk_group_root: Path) -> list:
    '''Point VIMEO_ROOT/sequences at exactly this chunk's group folders via
    symlinks - no copying, so this costs no extra disk beyond the chunk
    itself. Returns the list of group directory names now available.
    '''
    sequences_dir = VIMEO_ROOT / 'sequences'
    if sequences_dir.exists() or sequences_dir.is_symlink():
        shutil.rmtree(sequences_dir, ignore_errors=True)
    sequences_dir.mkdir(parents=True)

    group_names = []
    for group_dir in sorted(p for p in chunk_group_root.iterdir() if p.is_dir()):
        (sequences_dir / group_dir.name).symlink_to(group_dir, target_is_directory=True)
        group_names.append(group_dir.name)
    return group_names


def discover_complete_sequence_ids(sequences_dir: Path) -> list:
    '''List "<group>/<clip>" ids that have all 7 im*.png frames present.'''
    ids = []
    for group_dir in sorted(p for p in sequences_dir.iterdir() if p.is_dir()):
        for clip_dir in sorted(p for p in group_dir.iterdir() if p.is_dir()):
            if all((clip_dir / f'im{i}.png').is_file() for i in range(1, 8)):
                ids.append(f'{group_dir.name}/{clip_dir.name}')
    return ids


def write_chunk_split_lists(vimeo_root: Path, sequence_ids: list, seed: int, test_fraction: float = 0.1) -> None:
    '''(Re)write sep_trainlist.txt / sep_testlist.txt scoped to exactly the
    sequence ids available from the chunk currently linked into
    vimeo_root/sequences. nvc.data.vimeo treats these two files as
    authoritative and never re-splits them itself - something has to
    produce them, and this always overwrites the previous chunk's lists
    (correct here, since sequences/ was just replaced too).

    This is a per-chunk *training-progress* split, not a persistent
    benchmark - it is discarded with the chunk. It does NOT reproduce the
    official Vimeo-90K train/test split (that requires every chunk merged
    at once). Use DAVIS's existing test split for the actual final
    comparison between models trained by different people.
    '''
    ordered = sorted(sequence_ids)
    shuffled = ordered.copy()
    random.Random(seed).shuffle(shuffled)
    n_test = max(1, int(len(shuffled) * test_fraction))
    test_ids = sorted(shuffled[:n_test])
    train_ids = sorted(shuffled[n_test:])

    (vimeo_root / 'sep_trainlist.txt').write_text('\n'.join(train_ids) + '\n', encoding='utf-8')
    (vimeo_root / 'sep_testlist.txt').write_text('\n'.join(test_ids) + '\n', encoding='utf-8')
    print(f'[split] {len(train_ids)} train / {len(test_ids)} test sequence ids for this chunk')


def build_chunk_manifests(vimeo_root: Path, seed: int):
    train_path = defaults.vimeo_manifest_path.with_stem(defaults.vimeo_manifest_path.stem + '_train')
    test_path = defaults.vimeo_manifest_path.with_stem(defaults.vimeo_manifest_path.stem + '_test')
    build_sequence_manifest(vimeo_root, train_path, split='train', max_sequences=None, seed=seed, validate=True)
    build_sequence_manifest(vimeo_root, test_path, split='test', max_sequences=None, seed=seed, validate=True)
    return train_path, test_path


def load_progress() -> dict:
    if PROGRESS_PATH.is_file():
        return json.loads(PROGRESS_PATH.read_text(encoding='utf-8'))
    return {'completed_chunks': [], 'best_val_loss': None}


def save_progress(progress: dict) -> None:
    PROGRESS_PATH.write_text(json.dumps(progress, indent=2), encoding='utf-8')

## 6. Model, optimizer, resume from the last checkpoint if one exists

In [ ]:
model = BaselineAutoencoder(latent_channels=latent_channels).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
model_config = model.config_dict()

latest_ckpt = CHECKPOINT_DIR / 'latest.pt'
history = []
if latest_ckpt.is_file():
    epoch, history = resume_training_state(latest_ckpt, model=model, optimizer=optimizer, map_location=device)
    print(f'[resume] Loaded {latest_ckpt} - continuing from epoch {epoch}, {len(history)} epoch(s) of prior history')
else:
    epoch = 1
    print('[resume] No existing checkpoint on Drive - starting fresh')

progress = load_progress()
best_val_loss = progress['best_val_loss'] if progress['best_val_loss'] is not None else float('inf')
completed_so_far = progress['completed_chunks']
print(f'Already-completed chunks: {completed_so_far}')

## 7. Main loop: download -> train -> checkpoint -> delete -> next chunk

Safe to re-run after a disconnect - already-completed chunks are skipped, and training continues from the last saved checkpoint.

In [ ]:
for chunk_number in CHUNK_NUMBERS:
    if chunk_number in progress['completed_chunks']:
        print(f'[chunk {chunk_number}] already completed - skipping')
        continue

    sep = '=' * 60
    print()
    print(sep)
    print(f'[chunk {chunk_number}] starting')
    print(sep)

    try:
        chunk_group_root = download_and_extract_chunk(chunk_number, LOCAL_SCRATCH_DIR)
        group_names = relink_sequences_to_chunk(chunk_group_root)
        print(f'[chunk {chunk_number}] linked {len(group_names)} group folder(s)')

        sequence_ids = discover_complete_sequence_ids(VIMEO_ROOT / 'sequences')
        print(f'[chunk {chunk_number}] {len(sequence_ids)} complete 7-frame sequences found')
        if not sequence_ids:
            raise RuntimeError('no complete sequences found in this chunk')

        write_chunk_split_lists(VIMEO_ROOT, sequence_ids, seed=SEED)
        train_manifest, test_manifest = build_chunk_manifests(VIMEO_ROOT, seed=SEED)

        train_loader = create_sequence_train_loader(
            train_manifest, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, seed=SEED, crop_size=CROP_SIZE,
        )
        test_loader = create_sequence_test_loader(
            test_manifest, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, crop_size=CROP_SIZE,
        )

        for _ in range(EPOCHS_PER_CHUNK):
            train_metrics = train_one_epoch(model, train_loader, optimizer, device)
            val_metrics = validate_one_epoch(model, test_loader, device)

            train_loss = train_metrics['loss']
            val_loss = val_metrics['loss']
            val_psnr = val_metrics['psnr']

            history.append({
                'epoch': epoch, 'chunk': chunk_number,
                'train_loss': train_loss, 'val_loss': val_loss, 'val_psnr': val_psnr,
            })
            print(f'[chunk {chunk_number}] epoch {epoch}: train_mse={train_loss:.6f} val_mse={val_loss:.6f} val_psnr={val_psnr:.2f} dB')

            save_checkpoint(CHECKPOINT_DIR / 'latest.pt', model=model, optimizer=optimizer,
                             epoch=epoch, history=history, model_config=model_config)
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_path = CHECKPOINT_DIR / 'best.pt'
                save_checkpoint(best_path, model=model, optimizer=optimizer,
                                 epoch=epoch, history=history, model_config=model_config)
                print(f'  [best] new best val MSE {best_val_loss:.6f} -> {best_path}')
            (CHECKPOINT_DIR / 'history.json').write_text(json.dumps(history, indent=2), encoding='utf-8')

            epoch += 1

        progress['completed_chunks'].append(chunk_number)
        progress['best_val_loss'] = best_val_loss
        save_progress(progress)
        print(f'[chunk {chunk_number}] done, progress saved')

    except Exception as exc:
        print(f'[chunk {chunk_number}] FAILED: {exc!r} - not marked complete, re-run this cell to retry it')
    finally:
        if DELETE_CHUNK_AFTER_TRAINING and LOCAL_SCRATCH_DIR.exists():
            shutil.rmtree(LOCAL_SCRATCH_DIR)

print()
print('All requested chunks processed (or already were).')

## 8. Download the trained model to your laptop

Run this any time you want a copy - it does not need the loop above to be fully finished.

In [ ]:
from google.colab import files

print('Checkpoints on Drive:')
for p in sorted(CHECKPOINT_DIR.glob('*.pt')):
    size_mb = p.stat().st_size / 1e6
    print(' ', p, f'({size_mb:.1f} MB)')

files.download(str(CHECKPOINT_DIR / 'best.pt'))